# FloodNet segmentation: preprocessing, training, and evaluation

Run top to bottom. The notebook makes separate half-resolution copies, logs unreadable or unmatched pairs, audits possible test duplicates, trains the ResNet-34 U-Net and ResNet-18 baseline, tunes on validation only, then evaluates test once.

## Colab data setup

Select **Runtime → Change runtime type → GPU**. For Dropbox, ZIP the `FloodNet-Supervised_v1.0` and `ColorMasks-FloodNetv1.0` folders (including `ColorPalette-Values.xlsx` inside the latter), upload the ZIP, and paste its shared download link in the setup cell. The two folders may be ZIP siblings at the top level or nested together under a wrapper folder. The ZIP is extracted to temporary `/content` storage. Colab mounts Drive only for rolling training checkpoints; metrics and predictions remain in the runtime unless downloaded.



In [2]:
# Install SMP; Colab supplies its own GPU-enabled torch.
%pip -q install segmentation-models-pytorch tqdm

from pathlib import Path
import shutil, urllib.request, urllib.parse, zipfile
from pathlib import PurePosixPath
from tqdm.auto import tqdm

DROPBOX_ZIP_URL = "https://www.dropbox.com/scl/fo/k33qdif15ns2qv2jdxvhx/ANGaa8iPRhvlrvcKXjnmNRc?rlkey=ao2493wzl1cltonowjdbrnp7f&e=3&dl=1"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/Project_3")
    CHECKPOINT_ROOT = Path("/content/drive/MyDrive/DS4002/Project 3 (Lyle)/Colab/checkpoints")
else:
    ROOT = Path.cwd()
    if ROOT.name == "Scripts" and ROOT.parent.name == "Project_3": ROOT = ROOT.parent
    elif ROOT.name != "Project_3": ROOT /= "Project_3"
    CHECKPOINT_ROOT = ROOT / "checkpoints"
ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent checkpoints:", CHECKPOINT_ROOT)
DATA_ROOT = ROOT / "Data" / "FloodNet"

if "PASTE_YOUR_DROPBOX" in DROPBOX_ZIP_URL: raise ValueError("Paste a Dropbox ZIP shared link in DROPBOX_ZIP_URL.")
parts = urllib.parse.urlsplit(DROPBOX_ZIP_URL)
query = urllib.parse.parse_qs(parts.query); query["dl"] = ["1"]
url = urllib.parse.urlunsplit((*parts[:3], urllib.parse.urlencode(query,doseq=True), parts.fragment))
archive_path, extract_root = ROOT/"FloodNet.zip", ROOT/"dropbox_extract"
extract_root.mkdir(exist_ok=True)
if archive_path.exists() and archive_path.stat().st_size:
    print(f"Reusing downloaded archive ({archive_path.stat().st_size/1e9:.2f} GB).")
else:
    print("Downloading FloodNet ZIP from Dropbox...")
    with urllib.request.urlopen(url) as response, archive_path.open("wb") as output:
        total = int(response.headers.get("Content-Length", 0)) or None
        with tqdm(total=total, unit="B", unit_scale=True, desc="Download") as bar:
            while chunk := response.read(1024*1024):
                output.write(chunk); bar.update(len(chunk))
print(f"Checking ZIP ({archive_path.stat().st_size/1e9:.2f} GB)...")
with zipfile.ZipFile(archive_path) as archive:
    # Ignore macOS Finder metadata; normalize ZIP paths and reject traversal entries.
    members = [m for m in archive.infolist()
               if m.filename.replace("\\", "/") not in {"", "/"}
               and not m.filename.replace("\\", "/").startswith("__MACOSX/")]
    base = extract_root.resolve()
    targets, unsafe = [], []
    for member in members:
        rel = PurePosixPath(member.filename.replace("\\", "/"))
        target = extract_root.joinpath(*rel.parts)
        if rel.is_absolute() or ".." in rel.parts or not target.resolve().is_relative_to(base): unsafe.append(member.filename)
        else: targets.append((member, target))
    if unsafe: raise ValueError(f"Unsafe ZIP paths (showing up to 5): {unsafe[:5]}")
    print(f"Extracting {len(targets):,} entries ({sum(m.file_size for m,_ in targets)/1e9:.2f} GB uncompressed)...")
    with tqdm(total=sum(m.file_size for m,_ in targets), unit="B", unit_scale=True, desc="Extract") as bar:
        for member, target in targets:
            if member.is_dir(): target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as output:
                while chunk := source.read(1024*1024): output.write(chunk); bar.update(len(chunk))
print("Extraction complete. Locating FloodNet folders...")
image_roots = list(extract_root.rglob("FloodNet-Supervised_v1.0"))
if len(image_roots)!=1: raise FileNotFoundError("ZIP must contain one FloodNet-Supervised_v1.0 folder")
source_data = image_roots[0].parent  # accepts the two dataset folders as siblings, with or without a wrapper
mask_root = source_data/"ColorMasks-FloodNetv1.0"
if not (mask_root/"ColorPalette-Values.xlsx").is_file(): raise FileNotFoundError("Put ColorMasks-FloodNetv1.0 beside FloodNet-Supervised_v1.0, with the palette workbook inside")
print("Copying dataset into the notebook workspace...")
shutil.copytree(source_data, DATA_ROOT, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".DS_Store","processed_half"))
print(f"Data ready: {DATA_ROOT}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent checkpoints: /content/drive/MyDrive/DS4002/Project 3 (Lyle)/Colab/checkpoints
Reusing downloaded archive (13.06 GB).
Checking ZIP (13.06 GB)...
Extracting 7,045 entries (13.06 GB uncompressed)...


Extract:   0%|          | 0.00/13.1G [00:00<?, ?B/s]

Extraction complete. Locating FloodNet folders...
Copying dataset into the notebook workspace...
Data ready: /content/Project_3/Data/FloodNet


In [3]:
# If needed, install dependencies in the active notebook environment:
# %pip install torch torchvision segmentation-models-pytorch pandas matplotlib pillow

import csv, json, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image, ImageEnhance
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

SEED = 4002
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

if "ROOT" not in globals() or not str(ROOT).startswith("/content/Project_3"):
    ROOT = Path.cwd()
    if ROOT.name == "Scripts" and ROOT.parent.name == "Project_3": ROOT = ROOT.parent
    elif ROOT.name != "Project_3": ROOT = ROOT / "Project_3"
DATA_ROOT = ROOT / "Data" / "FloodNet"
RAW_ROOT = DATA_ROOT / "FloodNet-Supervised_v1.0"
HALF_ROOT = DATA_ROOT / "processed_half"
RESULTS_ROOT = ROOT / "Results" / "segmentation"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT = RESULTS_ROOT / "preprocessing_logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train": ("train/train-org-img", "train/train-label-img", "ColorMasks-TrainSet"),
    "val": ("val/val-org-img", "val/val-label-img", "ColorMasks-ValSet"),
    "test": ("test/test-org-img", "test/test-label-img", "ColorMasks-TestSet"),
}


Device: cuda


## Half-resolution data and labels

The setup writes resized copies under `processed_half` (bilinear images, nearest-neighbor masks), leaving originals unchanged. The supplied workbook defines the 10 RGB classes. Unreadable files and unmatched masks are excluded and logged.


In [4]:
# Official FloodNet RGB palette, in class-ID order.
CLASS_COLORS = {
    "background": (0,0,0), "building-flooded": (255,0,0),
    "building-non-flooded": (180,120,120), "road-flooded": (160,150,20),
    "road-non-flooded": (140,140,140), "water": (61,230,250),
    "tree": (0,82,255), "vehicle": (255,0,245), "pool": (255,235,0), "grass": (4,250,7),
}
CLASS_NAMES, NUM_CLASSES = list(CLASS_COLORS), len(CLASS_COLORS)

FILE_ISSUES = []
_PAIR_CACHE = {}

def get_pairs(split):
    if split not in _PAIR_CACHE:
        image_rel, label_rel, color_dir = SPLITS[split]
        image_dir = RAW_ROOT / image_rel
        mask_dir = DATA_ROOT / "ColorMasks-FloodNetv1.0" / color_dir
        if not mask_dir.is_dir(): mask_dir = RAW_ROOT / label_rel
        pairs = []
        for image in tqdm(sorted(image_dir.glob("*")), desc=f"Checking {split} pairs", leave=False):
            if image.suffix.lower() not in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}: continue
            mask = next((p for p in (mask_dir/f"{image.stem}_lab.png", mask_dir/f"{image.stem}.png") if p.is_file()), None)
            try:
                if mask is None: raise FileNotFoundError("mask missing")
                with Image.open(image) as im: im.verify()
                with Image.open(mask) as im: im.verify()
                pairs.append((image.stem, image, mask))
            except Exception as exc:
                FILE_ISSUES.append({"split":split,"image":str(image),"mask":str(mask or ""),"reason":str(exc)})
        _PAIR_CACHE[split] = pairs
    return _PAIR_CACHE[split]

def ensure_half_resolution():
    counts = {}
    for split in SPLITS:
        pairs = get_pairs(split); failed = set()
        for image_id, image_path, mask_path in tqdm(pairs, desc=f"Resizing {split}", unit="image", leave=False):
            out_image = HALF_ROOT / split / "images" / f"{image_id}.jpg"
            out_mask = HALF_ROOT / split / "masks" / f"{image_id}_lab.png"
            if out_image.exists() and out_mask.exists(): continue
            out_image.parent.mkdir(parents=True, exist_ok=True); out_mask.parent.mkdir(parents=True, exist_ok=True)
            try:
                with Image.open(image_path) as im:
                    size=(max(1,im.width//2),max(1,im.height//2))
                    im.convert("RGB").resize(size,Image.Resampling.BILINEAR).save(out_image,quality=95)
                with Image.open(mask_path) as im: im.resize(size,Image.Resampling.NEAREST).save(out_mask)
            except Exception as exc:
                failed.add(image_id); FILE_ISSUES.append({"split":split,"image":str(image_path),"mask":str(mask_path),"reason":f"resize failure: {exc}"})
                for path in (out_image,out_mask): path.unlink(missing_ok=True)
        _PAIR_CACHE[split] = [row for row in pairs if row[0] not in failed]
        counts[split] = len(_PAIR_CACHE[split])
    with (LOG_ROOT/"unreadable_or_unmatched.csv").open("w",newline="",encoding="utf-8") as f:
        writer=csv.DictWriter(f,fieldnames=["split","image","mask","reason"]); writer.writeheader(); writer.writerows(FILE_ISSUES)
    print("Readable pairs:", counts, "| excluded/logged:", len(FILE_ISSUES))
    return counts


OFFICIAL_COUNTS = ensure_half_resolution()
# Screen test images against train/validation with a 64-bit difference hash.
def dhash(path, size=8):
    with Image.open(path) as im: gray=im.convert("L").resize((size+1,size),Image.Resampling.BILINEAR)
    bits=np.asarray(gray)[:,1:] > np.asarray(gray)[:,:-1]
    return sum(int(bit)<<i for i,bit in enumerate(bits.ravel()))
refs=[]
for split in ("train","val"):
    refs.extend((dhash(path),image_id,split) for image_id,path,_ in tqdm(get_pairs(split), desc=f"Hashing {split}", unit="image", leave=False))
duplicate_rows=[]
for image_id,path,_ in tqdm(get_pairs("test"), desc="Auditing test duplicates", unit="image"):
    h=dhash(path); distance,ref_id,ref_split=min(((h^rh).bit_count(),rid,sp) for rh,rid,sp in refs)
    duplicate_rows.append({"test_id":image_id,"flagged":distance<=8,"closest_id":ref_id,"closest_split":ref_split,"distance":distance})
pd.DataFrame(duplicate_rows).to_csv(LOG_ROOT/"test_near_duplicates.csv",index=False)
print("Near-duplicate flags:",sum(r["flagged"] for r in duplicate_rows))
EXPECTED_COUNTS = {"train": 1445, "val": 450, "test": 448}
if OFFICIAL_COUNTS != EXPECTED_COUNTS: print(f"Count check: expected {EXPECTED_COUNTS}; usable pairs {OFFICIAL_COUNTS}")


Checking train pairs:   0%|          | 0/1445 [00:00<?, ?it/s]

Resizing train:   0%|          | 0/1445 [00:00<?, ?image/s]

Checking val pairs:   0%|          | 0/450 [00:00<?, ?it/s]

Resizing val:   0%|          | 0/450 [00:00<?, ?image/s]

Checking test pairs:   0%|          | 0/448 [00:00<?, ?it/s]

Resizing test:   0%|          | 0/448 [00:00<?, ?image/s]

Readable pairs: {'train': 1445, 'val': 450, 'test': 448} | excluded/logged: 0


Hashing train:   0%|          | 0/1445 [00:00<?, ?image/s]

Hashing val:   0%|          | 0/450 [00:00<?, ?image/s]

Auditing test duplicates:   0%|          | 0/448 [00:00<?, ?image/s]

Near-duplicate flags: 146


## Tiles and augmentation

Training uses random 512 × 512 crops with paired flips, 90° rotations, brightness jitter, and ImageNet normalization. Validation and test use overlapping whole-image windows without augmentation.


In [5]:
TILE = 512
STRIDE = 384  # overlapping inference windows
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


PALETTE_RGB = np.asarray(list(CLASS_COLORS.values()), dtype=np.int16)
MASK_COLOR_WARNINGS = set()

def encode_mask(mask_image, max_color_distance=100):
    rgb = np.asarray(mask_image.convert("RGB"), dtype=np.uint8)
    encoded = np.full(rgb.shape[:2], 255, dtype=np.uint8)
    for class_id, color in enumerate(PALETTE_RGB): encoded[np.all(rgb == color, axis=-1)] = class_id
    for color in np.unique(rgb[encoded == 255].reshape(-1, 3), axis=0):
        distance = np.linalg.norm(PALETTE_RGB - color.astype(np.int16), axis=1)
        class_id = int(distance.argmin())
        if distance[class_id] > max_color_distance:
            raise ValueError(f"Unrecognized mask color {tuple(color)} (nearest class {CLASS_NAMES[class_id]}, distance {distance[class_id]:.1f})")
        color = tuple(map(int, color))
        if color not in MASK_COLOR_WARNINGS:
            print(f"Mask color {color} mapped to nearest class '{CLASS_NAMES[class_id]}' (RGB distance {distance[class_id]:.1f})")
            MASK_COLOR_WARNINGS.add(color)
        encoded[np.all(rgb == color, axis=-1)] = class_id
    return encoded.astype(np.int64)


def image_tensor_array(image):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return ((arr - MEAN) / STD).transpose(2, 0, 1).astype(np.float32)

class FloodNetTiles(Dataset):
    def __init__(self, split, augment=False):
        self.split, self.augment = split, augment
        self.items = []
        for image_id, _, _ in tqdm(get_pairs(split), desc=f"Preparing {split} tiles", unit="image", leave=False):
            ip = HALF_ROOT / split / "images" / f"{image_id}.jpg"
            mp = HALF_ROOT / split / "masks" / f"{image_id}_lab.png"
            if ip.exists() and mp.exists(): self.items.append((image_id, ip, mp))
        if not self.items: raise RuntimeError(f"No resized pairs found for {split}")
    def __len__(self): return len(self.items)
    def __getitem__(self, index):
        image_id, ip, mp = self.items[index]
        with Image.open(ip) as im: image = im.convert("RGB")
        with Image.open(mp) as mm: mask = mm.copy()
        w, h = image.size
        if w < TILE or h < TILE: raise ValueError(f"{image_id} is smaller than {TILE}: {image.size}")
        x, y = (random.randint(0, w-TILE), random.randint(0, h-TILE)) if self.augment else ((w-TILE)//2, (h-TILE)//2)
        box = (x, y, x+TILE, y+TILE)
        image, mask = image.crop(box), mask.crop(box)
        if self.augment:
            if random.random() < .5: image, mask = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT), mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            if random.random() < .5: image, mask = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM), mask.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            k = random.randrange(4)
            if k: image, mask = image.rotate(90*k), mask.rotate(90*k)
            image = ImageEnhance.Brightness(image).enhance(random.uniform(.8, 1.2))
        return torch.from_numpy(image_tensor_array(image)), torch.from_numpy(encode_mask(mask)), image_id

train_ds = FloodNetTiles("train", augment=True)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=(DEVICE.type == "cuda"))
print("Training tiles per epoch:", len(train_ds))


Preparing train tiles:   0%|          | 0/1445 [00:00<?, ?image/s]

Training tiles per epoch: 1445


## Models and metrics

Both U-Nets use ImageNet encoders and the same cross-entropy + soft Dice loss. Whole-image window logits are averaged before computing IoU.


In [6]:
def make_model(encoder_name):
    return smp.Unet(encoder_name=encoder_name, encoder_weights="imagenet", in_channels=3,
                    classes=NUM_CLASSES, activation=None)


dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True, ignore_index=255,
                               classes=list(range(1, NUM_CLASSES)))
def combined_loss(logits, target):
    return nn.functional.cross_entropy(logits, target, ignore_index=255) + dice_loss(logits, target)


def confusion_add(confusion, truth, pred):
    valid = (truth >= 0) & (truth < NUM_CLASSES)
    inds = NUM_CLASSES * truth[valid].astype(np.int64) + pred[valid].astype(np.int64)
    confusion += np.bincount(inds, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    return confusion


def scores_from_confusion(confusion):
    tp = np.diag(confusion).astype(float)
    union = confusion.sum(0) + confusion.sum(1) - tp
    iou = np.divide(tp, union, out=np.full(NUM_CLASSES, np.nan), where=union > 0)
    return {"miou": float(np.nanmean(iou)), "per_class_iou": iou}


def sliding_positions(length, window=TILE, stride=STRIDE):
    if length <= window: return [0]
    xs = list(range(0, length-window+1, stride))
    if xs[-1] != length-window: xs.append(length-window)
    return xs

@torch.inference_mode()
def predict_whole_image(model, image_path, window=TILE, stride=STRIDE):
    with Image.open(image_path) as im: image = im.convert("RGB")
    w,h=image.size; pw,ph=max(0,window-w),max(0,window-h)
    if pw or ph:
        arr=np.asarray(image); mode="reflect" if min(arr.shape[:2])>1 else "edge"
        image=Image.fromarray(np.pad(arr,((0,ph),(0,pw),(0,0)),mode=mode)); w,h=image.size
    coords=[(y,x) for y in sliding_positions(h,window,stride) for x in sliding_positions(w,window,stride)]
    crops=[image_tensor_array(image.crop((x,y,x+window,y+window))) for y,x in coords]
    total=np.zeros((NUM_CLASSES,h,w),np.float32); count=np.zeros((h,w),np.float32)
    for start in range(0,len(crops),8):
        logits=model(torch.from_numpy(np.stack(crops[start:start+8])).to(DEVICE)).cpu().numpy()
        for logit,(y,x) in zip(logits,coords[start:start+8]):
            total[:,y:y+window,x:x+window]+=logit; count[y:y+window,x:x+window]+=1
    return (total/np.maximum(count,1))[:h-ph,:w-pw].argmax(0)

FLOODED_IDS = np.array([CLASS_NAMES.index(n) for n in ("building-flooded", "road-flooded", "water")])
BACKGROUND_ID = CLASS_NAMES.index("background")

def percent_flooded(mask):
    valid = mask != BACKGROUND_ID
    return 100*np.isin(mask[valid], FLOODED_IDS).sum()/valid.sum() if valid.any() else np.nan

@torch.inference_mode()
def evaluate_split(model, split, prediction_dir=None):
    model.eval(); confusion=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=np.int64); image_stats=[]
    items=get_pairs(split)
    if prediction_dir: prediction_dir.mkdir(parents=True,exist_ok=True)
    for image_id,_,_ in tqdm(items,desc=f"{split} evaluation",unit="image",leave=False):
        ip=HALF_ROOT/split/"images"/f"{image_id}.jpg"
        mp=HALF_ROOT/split/"masks"/f"{image_id}_lab.png"
        with Image.open(mp) as mm: truth=encode_mask(mm).astype(np.uint8)
        pred=predict_whole_image(model,ip).astype(np.uint8)
        if pred.shape!=truth.shape: raise ValueError(f"Size mismatch for {image_id}: {pred.shape} vs {truth.shape}")
        confusion_add(confusion,truth,pred)
        image_stats.append({"image_id":image_id,"predicted_percent_flooded":percent_flooded(pred),
                            "ground_truth_percent_flooded":percent_flooded(truth)})
        if prediction_dir: np.save(prediction_dir/f"{image_id}.npy",pred)
        del truth,pred
    print(f"{split}: evaluated {len(items)}/{len(items)} images")
    return scores_from_confusion(confusion),confusion,image_stats


## Validation tuning and resumable checkpoints

Each encoder is tested at learning rates 1e-4 and 3e-4, with patience 5 and a 40-epoch cap. After every epoch, a rolling full-state checkpoint is saved to Drive (model, optimizer, epoch, early-stopping and RNG state); interrupted runs resume automatically. The best validation weights are kept separately.


In [ ]:
LR_GRID = [1e-4, 3e-4]
MAX_EPOCHS = 40
PATIENCE = 5
val_pairs = get_pairs("val")

def validate_miou(model):
    return evaluate_split(model, "val")[0]["miou"]


RESUME_CHECKPOINTS = True

def save_checkpoint(path, state):
    temp = path.with_suffix(".tmp")
    torch.save(state, temp)
    temp.replace(path)

def fit_one(encoder_name, lr):
    model = make_model(encoder_name).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    run_dir = CHECKPOINT_ROOT / f"{encoder_name}_lr{lr:.0e}"
    run_dir.mkdir(parents=True, exist_ok=True)
    latest, best = run_dir/"latest.pt", run_dir/"best.pt"
    best_score, best_epoch, stale, history, start = -1.0, 0, 0, [], 1
    if RESUME_CHECKPOINTS and latest.exists():
        state = torch.load(latest, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"])
        best_score, best_epoch, stale = state["best_score"], state["best_epoch"], state["stale"]
        history, start = state["history"], state["epoch"]+1
        random.setstate(state["python_rng"]); np.random.set_state(state["numpy_rng"]); torch.set_rng_state(state["torch_rng"])
        if torch.cuda.is_available() and state["cuda_rng"]: torch.cuda.set_rng_state_all(state["cuda_rng"])
        print(f"Resuming {encoder_name} lr={lr:g} at epoch {start}")
    if stale >= PATIENCE or start > MAX_EPOCHS:
        print(f"{encoder_name} lr={lr:g} already reached its stopping point")
        return {"encoder":encoder_name,"lr":lr,"best_epoch":best_epoch,"best_val_miou":best_score,"checkpoint":str(best)}, history

    for epoch in range(start, MAX_EPOCHS+1):
        model.train(); losses=[]
        for images, masks, _ in tqdm(train_loader, desc=f"{encoder_name} lr={lr:g} epoch {epoch}/{MAX_EPOCHS}", unit="batch", leave=False):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss=combined_loss(model(images),masks); loss.backward(); optimizer.step()
            losses.append(float(loss.detach().cpu()))
        val_miou=validate_miou(model)
        history.append({"encoder":encoder_name,"lr":lr,"epoch":epoch,"train_loss":float(np.mean(losses)),"val_miou":val_miou})
        if val_miou > best_score:
            best_score,best_epoch,stale=val_miou,epoch,0
            save_checkpoint(best,{"state_dict":model.state_dict(),"encoder":encoder_name,"lr":lr,"epoch":epoch,"val_miou":val_miou})
        else: stale+=1
        state={"model":model.state_dict(),"optimizer":optimizer.state_dict(),"epoch":epoch,
               "best_score":best_score,"best_epoch":best_epoch,"stale":stale,"history":history,
               "python_rng":random.getstate(),"numpy_rng":np.random.get_state(),"torch_rng":torch.get_rng_state(),
               "cuda_rng":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}
        save_checkpoint(latest,state)
        print(f"{encoder_name} lr={lr:g} epoch={epoch:02d}: loss={np.mean(losses):.4f}, val_mIoU={val_miou:.4f} (checkpoint saved)")
        if stale >= PATIENCE:
            print(f"Early stopping; best epoch {best_epoch}"); break
    return {"encoder":encoder_name,"lr":lr,"best_epoch":best_epoch,"best_val_miou":best_score,"checkpoint":str(best)}, history

# This cell runs four training jobs and is compute intensive.
RUN_TUNING = True
if RUN_TUNING:
    tuning_results, tuning_history = [], []
    for encoder in ("resnet34", "resnet18"):
        print(f"Starting/resuming encoder: {encoder}")
        for lr in LR_GRID:
            result, history = fit_one(encoder, lr)
            tuning_results.append(result); tuning_history.extend(history)
    tuning_table = pd.DataFrame(tuning_results)
    history_df = pd.DataFrame(tuning_history)
    display(tuning_table)
    tuning_table.to_csv(RESULTS_ROOT / "validation_tuning_summary.csv", index=False)
    history_df.to_csv(RESULTS_ROOT / "validation_training_history.csv", index=False)
else:
    print("Set RUN_TUNING=True to train and tune the models.")


Starting/resuming encoder: resnet34


resnet34 lr=0.0001 epoch 1/40:   0%|          | 0/181 [00:00<?, ?batch/s]

Mask color (120, 120, 70) mapped to nearest class 'road-flooded' (RGB distance 70.7)
Mask color (0, 163, 255) mapped to nearest class 'tree' (RGB distance 81.0)


val evaluation:   0%|          | 0/450 [00:00<?, ?image/s]

val: evaluated 450/450 images
resnet34 lr=0.0001 epoch=01: loss=2.0865, val_mIoU=0.2371 (checkpoint saved)


resnet34 lr=0.0001 epoch 2/40:   0%|          | 0/181 [00:00<?, ?batch/s]

... (3 outputs left out)

val evaluation:   0%|          | 0/450 [00:00<?, ?image/s]

val: evaluated 450/450 images
resnet34 lr=0.0001 epoch=03: loss=1.2837, val_mIoU=0.3004 (checkpoint saved)


resnet34 lr=0.0001 epoch 4/40:   0%|          | 0/181 [00:00<?, ?batch/s]

val evaluation:   0%|          | 0/450 [00:00<?, ?image/s]

## Freeze choices, then evaluate test once

The best validation configuration is selected independently for each encoder. After saving those choices, the following cell performs exactly one test evaluation per chosen model and saves test predictions, aggregate confusion matrices, and per-class metrics. Do not rerun this cell to make modeling choices.

In [ ]:
if not RUN_TUNING: raise RuntimeError("Run tuning before test evaluation")
TEST_LOCK = RESULTS_ROOT / "test_evaluation_done.json"
if TEST_LOCK.exists(): raise RuntimeError(f"Test evaluation already completed and locked: {TEST_LOCK}")
chosen = tuning_table.sort_values("best_val_miou", ascending=False).groupby("encoder", as_index=False).first()
chosen.to_csv(RESULTS_ROOT / "locked_model_choices.csv", index=False)

TEST_RESULTS = {}
for _, row in tqdm(chosen.iterrows(), total=len(chosen), desc="Final test models", unit="model"):
    checkpoint = torch.load(row["checkpoint"], map_location=DEVICE, weights_only=False)
    model = make_model(row["encoder"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    pred_dir = RESULTS_ROOT / row["encoder"] / "test_predictions"
    metrics, confusion, image_stats = evaluate_split(model, "test", prediction_dir=pred_dir)
    TEST_RESULTS[row["encoder"]] = {"metrics": metrics, "confusion": confusion, "image_stats": image_stats,
                                    "lr": float(row["lr"]), "best_epoch": int(row["best_epoch"])}
    pd.DataFrame(confusion, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_ROOT / f"{row['encoder']}_test_confusion.csv")
    model.to("cpu"); del model

metric_rows = []
for encoder, result in TEST_RESULTS.items():
    metric_rows.append({"encoder": encoder, "test_miou": result["metrics"]["miou"],
                        "learning_rate": result["lr"], "best_epoch": result["best_epoch"]})
    for name, value in zip(CLASS_NAMES, result["metrics"]["per_class_iou"]):
        metric_rows.append({"encoder": encoder, "class": name, "per_class_iou": value})
metric_table = pd.DataFrame(metric_rows)
display(metric_table)
metric_table.to_csv(RESULTS_ROOT / "test_iou_metrics.csv", index=False)
TEST_LOCK.write_text(json.dumps({"completed": True, "models": chosen[["encoder", "lr", "best_epoch", "best_val_miou"]].to_dict(orient="records")}, indent=2))


## Flood impact comparison

Flooded pixels are building-flooded, road-flooded, and water; percent flooded is their share of non-background pixels. Impact cutoffs are editable assumptions (low <10%, moderate 10–30%, high ≥30%).


In [ ]:
LOW_MAX_PERCENT = 10.0
MODERATE_MAX_PERCENT = 30.0

def percent_flooded(mask):
    valid = mask != BACKGROUND_ID
    denominator = int(valid.sum())
    return (100.0 * float(np.isin(mask[valid], FLOODED_IDS).sum()) / denominator) if denominator else np.nan

def impact_category(percent):
    if not np.isfinite(percent): return "undefined"
    if percent < LOW_MAX_PERCENT: return "low"
    if percent < MODERATE_MAX_PERCENT: return "moderate"
    return "high"

impact_rows = []
for encoder, result in TEST_RESULTS.items():
    for row in tqdm(result["image_stats"], desc=f"Flood statistics: {encoder}", unit="image", leave=False):
        pred_pct, true_pct = row["predicted_percent_flooded"], row["ground_truth_percent_flooded"]
        impact_rows.append({"encoder":encoder, **row,
                            "absolute_percent_error":abs(pred_pct-true_pct) if np.isfinite(pred_pct) and np.isfinite(true_pct) else np.nan,
                            "predicted_impact":impact_category(pred_pct), "ground_truth_impact":impact_category(true_pct)})
impact_df = pd.DataFrame(impact_rows)
impact_df.to_csv(RESULTS_ROOT / "test_flood_impact_by_image.csv", index=False)
impact_summary = (impact_df.groupby("encoder").agg(
    mean_predicted_flood_percent=("predicted_percent_flooded", "mean"),
    mean_ground_truth_flood_percent=("ground_truth_percent_flooded", "mean"),
    mean_absolute_percent_error=("absolute_percent_error", "mean"),
    impact_category_accuracy=("predicted_impact", lambda s: np.mean(s.to_numpy() == impact_df.loc[s.index, "ground_truth_impact"].to_numpy()))
).reset_index())
display(impact_summary)
impact_summary.to_csv(RESULTS_ROOT / "test_flood_impact_summary.csv", index=False)


In [ ]:
print("Preparing test-image prediction comparisons...")
# Visual comparison of both models on three test images.
test_examples = get_pairs("test")[:3]
example_ids = [row[0] for row in test_examples]
fig, axes = plt.subplots(len(example_ids), 4, figsize=(16, 4*len(example_ids)), squeeze=False)
CLASS_CMAP = ListedColormap([tuple(np.array(rgb) / 255.0) for rgb in CLASS_COLORS.values()])
for row_i, image_id in enumerate(example_ids):
    with Image.open(HALF_ROOT / "test" / "images" / f"{image_id}.jpg") as im: image = im.convert("RGB")
    _, _, mask_path = test_examples[row_i]
    with Image.open(HALF_ROOT / "test" / "masks" / f"{image_id}_lab.png") as mm: truth = encode_mask(mm).astype(np.uint8)
    p34 = np.load(RESULTS_ROOT / "resnet34" / "test_predictions" / f"{image_id}.npy")
    p18 = np.load(RESULTS_ROOT / "resnet18" / "test_predictions" / f"{image_id}.npy")
    panels = [(np.asarray(image), "Image"), (truth, "Ground truth"), (p34, "U-Net ResNet-34"), (p18, "Baseline ResNet-18")]
    for col, (arr, title) in enumerate(panels):
        axes[row_i, col].imshow(arr, cmap=None if col == 0 else CLASS_CMAP, vmin=0 if col else None, vmax=NUM_CLASSES-1 if col else None)
        axes[row_i, col].set_title(f"{title} · {image_id}"); axes[row_i, col].axis("off")
plt.tight_layout(); plt.show()


## Reporting

No cross-validation or significance test is used. The official split is preserved; validation selects configurations, and the locked test split is reported once. Outputs are saved under `Results/segmentation/`.
